In [35]:
import os, sys, importlib, pathlib
import pandas as pd
from pathlib import Path
sys.path.append (os.path.abspath(".."))
# my utils
from utils import fa
from utils import modeling
from utils import preprocess_listings

# 标准版存档V0：

LABELS_EN=[
            'open to different cultures', 'cosmopolitan','international view', 'cultural exchange',
            'personal life', 'life experiences', 'divers interests', 'hobbies', 'enjoy life',
            'meet new people', 'welcoming', 'friendly', 'sociable', 'interpersonal interaction',
            'thoughtful service', 'attentive to needs', 'willing to help', 'responsive',
            'fan of Airbnb', 'Airbnb community','love Airbnb', 'travel with Airbnb'
        ]
DICT_FACTOR_ITEMS={"ouverture":['open to different cultures', 'cosmopolitan','international view', 'cultural exchange'],
                   "authenticité":['personal life', 'life experiences', 'divers interests', 'hobbies', 'enjoy life'],
                   "sociabilité":['meet new people', 'welcoming', 'friendly', 'sociable', 'interpersonal interaction'],
                   "auto_promotion":['thoughtful service', 'attentive to needs', 'willing to help', 'responsive'],
                   "exemplarité":['fan of Airbnb', 'Airbnb community','love Airbnb', 'travel with Airbnb']}


In [2]:
path_df="../data_processed\listings_tactics_bio_vis-paris_london-2306_2406.csv"
df_all=pd.read_csv(path_df)
print(df_all.shape)

C:\Users\yeliu\AppData\Local\Temp\ipykernel_31072\2475269784.py:2: DtypeWarning: Columns (68) have mixed types. Specify dtype option on import or set low_memory=False.
  df_all=pd.read_csv(path_df)


(187851, 123)


In [36]:
# drop no pic match?
df=df_all.copy()
df=df[df["has_face"].notna()]
print(len(df_all),len(df))

187851 179316


In [40]:

vars_tactics=[
    "ouverture", "authenticité","sociabilité","auto_promotion","exemplarité", 
    "host_picture_type","is_smiling", "smile_score"
    # "age_class","gender",
    ]

df[vars_tactics].value_counts(dropna=False)
df[vars_tactics]=df[vars_tactics].fillna(0)


## ols:

In [ ]:
importlib.reload(modeling)
from utils.modeling import write_formula,build_model

import statsmodels.api as sm
import statsmodels.formula.api as smf


# C(host_is_superhost)
# C(host_has_profile_pic)
formula= ("booking_rate_l90d ~ C(host_identity_verified)  + C(lang) + C(room_type) + C(instant_bookable) + review_scores_rating + has_rating + years_since_host + professional_host + len + price + availability_90 + "
        "C(host_is_superhost) + "
        "(ouverture + authenticité + sociabilité + auto_promotion + exemplarité +  C(host_picture_type) + is_smiling ) * is_paris * in_2024 "
        # smile_score coef 过小!
        # "is_paris * in_2024"
        
)
model=smf.ols(formula, data=df).fit()
summary=model.summary()
print(summary)

                            OLS Regression Results                            
Dep. Variable:      booking_rate_l90d   R-squared:                       0.316
Model:                            OLS   Adj. R-squared:                  0.316
Method:                 Least Squares   F-statistic:                     1624.
Date:                Mon, 23 Mar 2026   Prob (F-statistic):               0.00
Time:                        22:15:21   Log-Likelihood:                 36307.
No. Observations:              179316   AIC:                        -7.251e+04
Df Residuals:                  179264   BIC:                        -7.198e+04
Df Model:                          51                                         
Covariance Type:            nonrobust                                         
                                                         coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------

In [58]:
# subdf=df[df['in_2024']==1]
# print(subdf[['is_paris','in_2024']].value_counts(dropna=False))
# formula= ("booking_rate_l90d ~ C(host_identity_verified)  + C(lang) + C(room_type) + C(instant_bookable) + + review_scores_rating + has_rating + years_since_host + professional_host + len + price + availability_90 + "
#         "C(host_is_superhost) + ouverture + authenticité + sociabilité + auto_promotion + exemplarité +  C(host_picture_type) + is_smiling + "
#         # smile_score coef 过小!
#         "is_paris "
#         # "in_2024"
        
# )
# model=smf.ols(formula, data=subdf).fit()
# summary=model.summary()
# print(summary)

In [68]:
formula= ("is_smiling ~ C(host_identity_verified) + C(host_has_profile_pic) + C(host_is_superhost) + C(lang) + C(room_type) + C(instant_bookable) + review_scores_rating + has_rating + years_since_host + professional_host + len + price + availability_90 + "
        # "ouverture + authenticité + sociabilité + auto_promotion + exemplarité + is_smiling +  C(host_picture_type)+"
        # smile_score coef 过小!
        "is_paris * in_2024"
)
model=smf.ols(formula, data=df).fit()
summary=model.summary()
print(summary)

                            OLS Regression Results                            
Dep. Variable:             is_smiling   R-squared:                       0.060
Model:                            OLS   Adj. R-squared:                  0.060
Method:                 Least Squares   F-statistic:                     569.9
Date:                Mon, 23 Mar 2026   Prob (F-statistic):               0.00
Time:                        22:12:28   Log-Likelihood:            -1.0712e+05
No. Observations:              179316   AIC:                         2.143e+05
Df Residuals:                  179295   BIC:                         2.145e+05
Df Model:                          20                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

In [6]:
importlib.reload(modeling)
from utils.modeling import modeling_main,build_model

#-------------------------------vars------------------------------------

x_vars=["host_identity_verified", "host_has_profile_pic",         
    "review_scores_rating", #"has_rating", "number_of_reviews",
    "years_since_host","professional_host",'host_is_superhost', ##'calculated_host_listings_count',
    # 'status_changed', 'old_host_sp_changed',# "is_changed"
    "lang", "len",
    "price",
    "availability_90",#*
    "room_type", "instant_bookable", #'is_within_1km',
    
    "ouverture", "authenticité","sociabilité","auto_promotion","exemplarité", 
    "host_picture_type","is_smiling"
]  
tactics_vars=["is_paris","in_2024"]

modeling_main(df_input=df_all, 
            x_vars=x_vars, 
            y_var="booking_rate_l90d", 
            key_vars=tactics_vars, 
            group_col=None,#"host_is_superhost',#==stauts_changed+sp_changed
            output_folder=None,
            to_fillna0=True,
            run_vif=True,
            # save_models_summary=False,  
            save_models_table=False,
            save_plots=False,
            ndigits=4
        )

=============================================check data=============================================
[INFO] fillna in key cols!`

========================================check output folder=========================================
================================================vif=================================================
+key_vars ; - group_cols
remove key_vars in x_vars
Final x_vars_ctrl :['host_identity_verified', 'host_has_profile_pic', 'review_scores_rating', 'years_since_host', 'professional_host', 'host_is_superhost', 'lang', 'len', 'price', 'availability_90', 'room_type', 'instant_bookable', 'ouverture', 'authenticité', 'sociabilité', 'auto_promotion', 'exemplarité', 'host_picture_type', 'is_smiling']

[INFO] formula :
 booking_rate_l90d ~ C(host_identity_verified) + C(host_has_profile_pic) + C(host_is_superhost) + C(lang) + C(room_type) + C(instant_bookable) + C(host_picture_type) + review_scores_rating + years_since_host + professional_host + len + price + availabilit

,Variables,VIF,Niveau_colinearite
0,Intercept,91561.521721,***
1,C(host_identity_verified)[T.t],1.017968,*
2,C(host_has_profile_pic)[T.t],1.000178,*
3,C(host_is_superhost)[T.t],1.095642,*
4,C(lang)[T.fr],1.926281,*
5,C(lang)[T.no_text],1.008223,*
6,C(lang)[T.other_langs],1.038891,*
7,C(room_type)[T.Hotel room],1.042722,*
8,C(room_type)[T.Private room],1.167226,*
9,C(room_type)[T.Shared room],1.007452,*



 ============================================basic model============================================= 

[NUMBER CHECK1] x_vars:19, x_vars_ctrl:19
['host_identity_verified', 'host_has_profile_pic', 'review_scores_rating', 'years_since_host', 'professional_host', 'host_is_superhost', 'lang', 'len', 'price', 'availability_90', 'room_type', 'instant_bookable', 'ouverture', 'authenticité', 'sociabilité', 'auto_promotion', 'exemplarité', 'host_picture_type', 'is_smiling', 'booking_rate_l90d']
-key_vars ; - group_cols
Final x_vars_ctrl :['host_identity_verified', 'host_has_profile_pic', 'review_scores_rating', 'years_since_host', 'professional_host', 'host_is_superhost', 'lang', 'len', 'price', 'availability_90', 'room_type', 'instant_bookable', 'ouverture', 'authenticité', 'sociabilité', 'auto_promotion', 'exemplarité', 'host_picture_type', 'is_smiling']

[INFO] formula :
 booking_rate_l90d ~ C(host_identity_verified) + C(host_has_profile_pic) + C(host_is_superhost) + C(lang) + C(room_type)

TypeError: 'in <string>' requires string as left operand, not NoneType